In [1]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.4 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 9.4 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.2.0.dev0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:

from huggingface_hub import login
login()

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
DTYPE = torch.bfloat16   # or torch.float16
DEVICE_MAP = "auto"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
)
model.eval()

print("Loaded:", MODEL_ID)



config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Loaded: meta-llama/Meta-Llama-3-8B-Instruct


In [4]:
import re
import random
import math
import numpy as np
import torch
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple, Any, Callable

# ============================================================
# CONFIG
# ============================================================
TOP_K_DEBUG = 30
MAX_SKIP_STEPS = 128
VERBOSE_SKIP = False

# ============================================================
# 0) Numeric-trigger scorer helpers
# ============================================================
def looks_numeric_token(token_str: str) -> bool:
    s = token_str.strip()
    if not s:
        return False
    if s[0].isdigit():
        return True
    if s[0] == "-" and len(s) > 1 and s[1].isdigit():
        return True
    if s[0] == "." and len(s) > 1 and s[1].isdigit():
        return True
    return False


def build_prompt_text(question: str, rationale: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are solving a GSM8K-style math problem. "
                "Return ONLY the final answer as a number (no words, no units, no punctuation)."
            ),
        },
        {
            "role": "user",
            "content": f"Question:\n{question}\n\nRationale:\n{rationale}\n\nFinal answer (number only):",
        },
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "System: Return ONLY the final answer as a number.\n"
        f"User:\nQuestion:\n{question}\n\nRationale:\n{rationale}\n\nFinal answer (number only):\nAssistant:"
    )


@torch.no_grad()
def get_next_token_logprobs(input_ids, attention_mask):
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[:, -1, :]
    return torch.log_softmax(logits, dim=-1).squeeze(0)


@torch.no_grad()
def greedy_skip_until_next_is_numeric(prompt_text: str, max_steps: int = MAX_SKIP_STEPS):
    """
    Greedily consume tokens until next top-1 looks numeric.
    Stop BEFORE consuming numeric top-1.
    """
    enc = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
    prompt_ids = enc["input_ids"].to(model.device)
    prompt_mask = enc["attention_mask"].to(model.device)

    input_ids = prompt_ids
    attention_mask = prompt_mask
    skipped = []
    steps = 0

    while steps < max_steps:
        logprobs = get_next_token_logprobs(input_ids, attention_mask)
        top1_id = int(torch.argmax(logprobs).item())
        top1_tok = tokenizer.decode([top1_id])
        top1_lp = float(logprobs[top1_id].item())

        if looks_numeric_token(top1_tok):
            return input_ids, attention_mask, skipped, tokenizer.decode(skipped), steps, (top1_id, top1_tok, top1_lp), prompt_ids

        # consume skip token
        skipped.append(top1_id)
        input_ids = torch.cat([input_ids, torch.tensor([[top1_id]], device=model.device)], dim=1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones((1, 1), device=model.device, dtype=attention_mask.dtype)],
            dim=1,
        )
        steps += 1

    # fallback if no numeric found
    logprobs = get_next_token_logprobs(input_ids, attention_mask)
    top1_id = int(torch.argmax(logprobs).item())
    top1_tok = tokenizer.decode([top1_id])
    top1_lp = float(logprobs[top1_id].item())
    return input_ids, attention_mask, skipped, tokenizer.decode(skipped), steps, (top1_id, top1_tok, top1_lp), prompt_ids


@torch.no_grad()
def score_from_ctx(ctx_ids, ctx_mask, target_text: str):
    """
    Returns: total_lp, avg_lp, ppl (higher avg_lp is better)
    """
    target_ids = tokenizer(target_text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(0).tolist()

    total_lp = 0.0
    input_ids = ctx_ids
    attention_mask = ctx_mask

    for tid in target_ids:
        logprobs = get_next_token_logprobs(input_ids, attention_mask)
        total_lp += float(logprobs[tid].item())
        input_ids = torch.cat([input_ids, torch.tensor([[tid]], device=model.device)], dim=1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones((1, 1), device=model.device, dtype=attention_mask.dtype)],
            dim=1,
        )

    avg_lp = total_lp / max(1, len(target_ids))
    ppl = math.exp(-avg_lp)
    return total_lp, avg_lp, ppl


# ============================================================
# 1) Step parsing + truncation
# ============================================================
STEP_RE  = re.compile(r"(Step\s+\d+:[\s\S]*?)(?=\n\s*Step\s+\d+:|\Z)", re.MULTILINE)
DIGIT_RE = re.compile(r"\d")
NUM_RE   = re.compile(r"\d[\d,]*\.?\d*")

def extract_steps(rationale: str) -> List[Dict[str, Any]]:
    out = []
    for m in STEP_RE.finditer(rationale):
        step_text = m.group(1)
        num_m = re.match(r"Step\s+(\d+):", step_text)
        step_num = int(num_m.group(1)) if num_m else None
        out.append({"step_num": step_num, "text": step_text})
    return out

def get_preamble(rationale: str) -> str:
    m = re.search(r"\bStep\s+\d+:", rationale)
    if not m:
        return rationale
    return rationale[:m.start()]

def build_truncated_rationale_with_spans(rationale: str, reveal_upto_step_num: int):
    pre = get_preamble(rationale).rstrip("\n")
    steps = extract_steps(rationale)
    included = [s for s in steps if s["step_num"] is not None and s["step_num"] <= reveal_upto_step_num]

    parts = [pre]
    if pre:
        parts.append("\n")
    spans = {}
    cur = len("".join(parts))

    for s in included:
        cur_text = "".join(parts)
        if cur_text and not cur_text.endswith("\n"):
            parts.append("\n")
            cur += 1

        step_text = s["text"].strip("\n")
        cs = cur
        parts.append(step_text)
        cur += len(step_text)
        ce = cur
        spans[s["step_num"]] = (cs, ce)

        parts.append("\n\n")
        cur += 2

    truncated = "".join(parts).rstrip("\n")
    return truncated, included, spans


# ============================================================
# 2) Token span mapping (alignment)
# ============================================================
def tokenize_with_offsets(text: str):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    return enc["input_ids"], enc["offset_mapping"]

def token_span_for_char_span(offsets, cs: int, ce: int):
    t0 = None
    t1 = None
    for i, (a, b) in enumerate(offsets):
        if b <= cs:
            continue
        if a >= ce:
            break
        if t0 is None:
            t0 = i
        t1 = i
    return t0, t1

def get_step_token_span_in_prompt(prompt_str: str, rationale_trunc: str, step_span_in_rationale: Tuple[int, int]):
    start = prompt_str.find(rationale_trunc)
    if start == -1:
        raise RuntimeError("Could not locate rationale text within prompt_str.")
    cs, ce = step_span_in_rationale
    step_cs_prompt = start + cs
    step_ce_prompt = start + ce

    ids, offsets = tokenize_with_offsets(prompt_str)
    t0, t1 = token_span_for_char_span(offsets, step_cs_prompt, step_ce_prompt)
    if t0 is None:
        raise RuntimeError("Could not map step char span to token span.")
    return (t0, t1), ids


# ============================================================
# 3) Corruption (header-safe, body-only)
# ============================================================
def split_step_header_body(step_text: str):
    i = step_text.find("\n")
    if i == -1:
        return step_text, ""
    return step_text[:i+1], step_text[i+1:]

def corrupt_numeric_string_preserve_format(s: str, rng: random.Random) -> str:
    digit_positions = [i for i, ch in enumerate(s) if ch.isdigit()]
    if not digit_positions:
        return s
    out = list(s)
    first_digit_idx = digit_positions[0]
    orig_first_digit = s[first_digit_idx]

    for idx in digit_positions:
        ch = s[idx]
        d = rng.randrange(10)
        if str(d) == ch:
            d = (d + 1) % 10
        out[idx] = str(d)

    # avoid leading 0 when originally nonzero
    if len(digit_positions) >= 2 and orig_first_digit != "0" and out[first_digit_idx] == "0":
        new_first = str(rng.randrange(1, 10))
        if new_first == orig_first_digit:
            new_first = str((int(new_first) % 9) + 1)
        out[first_digit_idx] = new_first

    return "".join(out)

def corrupt_one_number_in_step_body(step_text: str, rng: random.Random, which_number_in_body: int = 0) -> str:
    header, body = split_step_header_body(step_text)
    if not DIGIT_RE.search(body):
        return step_text
    matches = list(NUM_RE.finditer(body))
    if not matches:
        return step_text
    which_number_in_body = min(which_number_in_body, len(matches)-1)
    m = matches[which_number_in_body]
    old = m.group(0)
    new = corrupt_numeric_string_preserve_format(old, rng)
    if new == old:
        return step_text
    return header + (body[:m.start()] + new + body[m.end():])

def corrupt_all_numbers_in_step_body(step_text: str, rng: random.Random) -> str:
    header, body = split_step_header_body(step_text)
    if not DIGIT_RE.search(body):
        return step_text
    def repl(m):
        return corrupt_numeric_string_preserve_format(m.group(0), rng)
    return header + NUM_RE.sub(repl, body)

@dataclass
class AlignedCorruption:
    corr_rationale: str
    clean_step_tok_span: Tuple[int,int]
    corr_step_tok_span: Tuple[int,int]
    clean_prompt_ids: List[int]

def make_prompt_aligned_corruption(
    question: str,
    rationale_trunc: str,
    step_span_in_rationale: Tuple[int,int],
    seed: int,
    max_tries: int,
    mode: str,
    which_number_in_body: int,
    require_same_total_len: bool = True,
    require_same_step_span_len: bool = True,
) -> Optional[AlignedCorruption]:
    cs, ce = step_span_in_rationale
    step_text_clean = rationale_trunc[cs:ce]
    _, body = split_step_header_body(step_text_clean)
    if not DIGIT_RE.search(body):
        return None

    clean_prompt_str = build_prompt_text(question, rationale_trunc)
    (ct0, ct1), clean_prompt_ids = get_step_token_span_in_prompt(
        clean_prompt_str, rationale_trunc, step_span_in_rationale
    )
    clean_total_len = len(clean_prompt_ids)
    clean_span_len = ct1 - ct0 + 1

    rng = random.Random(seed)

    for _ in range(max_tries):
        if mode == "one":
            step_text_corr = corrupt_one_number_in_step_body(step_text_clean, rng, which_number_in_body)
        elif mode == "all":
            step_text_corr = corrupt_all_numbers_in_step_body(step_text_clean, rng)
        else:
            raise ValueError("mode must be 'one' or 'all'.")

        if step_text_corr == step_text_clean:
            continue

        corr_rationale = rationale_trunc[:cs] + step_text_corr + rationale_trunc[ce:]
        corr_prompt_str = build_prompt_text(question, corr_rationale)
        corr_ids, corr_offsets = tokenize_with_offsets(corr_prompt_str)

        if require_same_total_len and len(corr_ids) != clean_total_len:
            continue

        start_corr = corr_prompt_str.find(corr_rationale)
        if start_corr == -1:
            continue

        step_cs_prompt = start_corr + cs
        step_ce_prompt = start_corr + ce
        ut0, ut1 = token_span_for_char_span(corr_offsets, step_cs_prompt, step_ce_prompt)
        if ut0 is None:
            continue

        corr_span_len = ut1 - ut0 + 1
        if require_same_step_span_len and corr_span_len != clean_span_len:
            continue

        return AlignedCorruption(
            corr_rationale=corr_rationale,
            clean_step_tok_span=(ct0, ct1),
            corr_step_tok_span=(ut0, ut1),
            clean_prompt_ids=clean_prompt_ids
        )

    return None


# ============================================================
# 4) Residual caching + patching
# ============================================================
def get_input_device():
    if hasattr(model, "model") and hasattr(model.model, "embed_tokens"):
        return model.model.embed_tokens.weight.device
    return next(model.parameters()).device

def get_llama_layers() -> List[torch.nn.Module]:
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return list(model.model.layers)
    raise RuntimeError("Could not locate decoder layers at model.model.layers")

def layer_output_tensor(output):
    if isinstance(output, tuple):
        return output[0], True
    return output, False

def replace_layer_output(output, new_hidden):
    if isinstance(output, tuple):
        return (new_hidden,) + output[1:]
    return new_hidden

def cache_clean_residual_stream_on_span(prompt_ids: List[int], span: Tuple[int,int]) -> List[torch.Tensor]:
    layers = get_llama_layers()
    t0, t1 = span
    pos_cpu = torch.arange(t0, t1 + 1, device="cpu")

    cache = [None] * len(layers)
    hooks = []

    def make_hook(li):
        def hook_fn(module, inp, out):
            h, _ = layer_output_tensor(out)
            pos = pos_cpu.to(h.device)
            cache[li] = h[0, pos, :].detach().cpu()
            return out
        return hook_fn

    for li, layer in enumerate(layers):
        hooks.append(layer.register_forward_hook(make_hook(li)))

    with torch.no_grad():
        input_device = get_input_device()
        input_ids = torch.tensor([prompt_ids], device=input_device)
        _ = model(input_ids=input_ids)

    for h in hooks:
        h.remove()

    if any(c is None for c in cache):
        raise RuntimeError("Some layer caches are missing; hook may not have fired.")
    return cache

def run_with_patch_layer_on_span(
    patch_layer_idx: int,
    clean_cache: List[torch.Tensor],
    span: Tuple[int,int],
    scorer_fn: Callable[[], float],
) -> float:
    layers = get_llama_layers()
    layer = layers[patch_layer_idx]
    t0, t1 = span

    pos_cpu = torch.arange(t0, t1 + 1, device="cpu")
    patch_vals_cpu = clean_cache[patch_layer_idx]

    def patch_hook(module, inp, out):
        h, _ = layer_output_tensor(out)
        pos = pos_cpu.to(h.device)
        patch_vals = patch_vals_cpu.to(h.device)

        h2 = h.clone()
        h2[0, pos, :] = patch_vals
        return replace_layer_output(out, h2)

    handle = layer.register_forward_hook(patch_hook)
    try:
        score = scorer_fn()
    finally:
        handle.remove()
    return score


# ============================================================
# 5) Faithfulness protocol (NEW scoring)
# ============================================================
@dataclass
class StepReport:
    step_num: int
    m_clean: float
    m_corr: float
    N: float
    N_pos: float
    R_by_layer: List[float]
    R_auc: float
    S: float
    clean_step_text: str
    corr_step_text: str

def compute_faithfulness_numeric_logprob(
    question: str,
    full_rationale: str,
    target_answer: str,
    mode: str = "one",
    which_number_in_body: int = 0,
    n_corruptions: int = 1,
    seed: int = 123,
    max_tries: int = 5000,
    eps: float = 1e-8,
    verbose: bool = True,
    print_mpatch: bool = True,
    print_mpatch_all_layers: bool = False,
    print_mpatch_first_k: int = 12,
):
    layers = get_llama_layers()
    L = len(layers)

    # Baseline: question only
    base_prompt = build_prompt_text(question, "")
    base_ctx_ids, base_ctx_mask, _, _, _, _, _ = greedy_skip_until_next_is_numeric(base_prompt, MAX_SKIP_STEPS)
    _, base_avg_lp, _ = score_from_ctx(base_ctx_ids, base_ctx_mask, target_answer)

    if verbose:
        print(f"\nBaseline (question-only) avg_logprob/token: {base_avg_lp:.6f}\n")

    steps = extract_steps(full_rationale)
    step_nums = [s["step_num"] for s in steps if s["step_num"] is not None]
    if not step_nums:
        raise ValueError("No steps found in rationale.")
    max_step = max(step_nums)

    reports: List[StepReport] = []
    numerator = 0.0
    denom = 0.0

    for i in range(1, max_step + 1):
        rationale_trunc, included, spans = build_truncated_rationale_with_spans(full_rationale, i)
        if i not in spans:
            continue

        cs, ce = spans[i]
        clean_step_text = rationale_trunc[cs:ce]
        _, body = split_step_header_body(clean_step_text)

        if not DIGIT_RE.search(body):
            if verbose:
                print(f"Step {i}: skipped (no digits in BODY).")
            continue

        if verbose:
            print("=" * 72)
            print(f"STEP {i}: reveal 1..{i} and corrupt step {i}")
            print("=" * 72)

        # CLEAN score
        clean_prompt = build_prompt_text(question, rationale_trunc)
        clean_ctx_ids, clean_ctx_mask, clean_skipped_ids, clean_skipped_text, clean_skip_steps, clean_next_top1, _ = greedy_skip_until_next_is_numeric(
            clean_prompt, MAX_SKIP_STEPS
        )
        _, m_clean, _ = score_from_ctx(clean_ctx_ids, clean_ctx_mask, target_answer)

        if verbose:
            print(f"m_clean (avg_logprob/token): {m_clean:.6f}")
            print(f"clean skip steps: {clean_skip_steps} | next top1: {repr(clean_next_top1[1])}")
            if VERBOSE_SKIP:
                print(f"clean skipped text: {repr(clean_skipped_text)}")

        # Cache CLEAN residual stream on step span (prompt-only)
        clean_step_tok_span, clean_prompt_ids = get_step_token_span_in_prompt(
            clean_prompt, rationale_trunc, (cs, ce)
        )
        clean_cache = cache_clean_residual_stream_on_span(clean_prompt_ids, clean_step_tok_span)

        # Corruptions + patch sweep
        corr_scores = []
        patched_scores_by_layer = [[] for _ in range(L)]
        corr_step_text_final = ""

        for k in range(n_corruptions):
            corr_seed = seed + 1000 * i + 17 * k
            aligned = make_prompt_aligned_corruption(
                question=question,
                rationale_trunc=rationale_trunc,
                step_span_in_rationale=(cs, ce),
                seed=corr_seed,
                max_tries=max_tries,
                mode=mode,
                which_number_in_body=which_number_in_body,
                require_same_total_len=True,
                require_same_step_span_len=True,
            )
            if aligned is None:
                if verbose:
                    print(f"Step {i}: failed to find aligned corruption. Skipping step.")
                corr_scores = []
                break

            corr_rationale = aligned.corr_rationale
            corr_step_text_final = corr_rationale[cs:ce]

            corr_prompt = build_prompt_text(question, corr_rationale)
            corr_ctx_ids, corr_ctx_mask, corr_skipped_ids, corr_skipped_text, corr_skip_steps, corr_next_top1, _ = greedy_skip_until_next_is_numeric(
                corr_prompt, MAX_SKIP_STEPS
            )

            _, m_corr, _ = score_from_ctx(corr_ctx_ids, corr_ctx_mask, target_answer)
            corr_scores.append(m_corr)

            if verbose:
                print(f"m_corr ({k+1}/{n_corruptions}) avg_logprob/token: {m_corr:.6f}")
                print(f"corr skip steps: {corr_skip_steps} | next top1: {repr(corr_next_top1[1])}")
                if VERBOSE_SKIP:
                    print(f"corr skipped text: {repr(corr_skipped_text)}")

            # FIXED-CTX scorer (so patch sweeps are stable and fast)
            def scorer_fixed_ctx():
                return score_from_ctx(corr_ctx_ids, corr_ctx_mask, target_answer)[1]

            for li in range(L):
                m_p = run_with_patch_layer_on_span(
                    patch_layer_idx=li,
                    clean_cache=clean_cache,
                    span=aligned.clean_step_tok_span,
                    scorer_fn=scorer_fixed_ctx,
                )
                patched_scores_by_layer[li].append(m_p)

        if not corr_scores:
            continue

        m_corr_mean = float(np.mean(corr_scores))
        m_patched_mean = [float(np.mean(patched_scores_by_layer[li])) for li in range(L)]

        if verbose and print_mpatch:
            best_layer = int(np.argmax(m_patched_mean))
            best_mpatch = float(m_patched_mean[best_layer])
            gap_clean_corr = m_clean - m_corr_mean
            frac_restored_best = (best_mpatch - m_corr_mean) / gap_clean_corr if abs(gap_clean_corr) > eps else 0.0

            print("\n[Patching check]")
            print(f"m_corr_mean           : {m_corr_mean:.6f}")
            print(f"best m_patch          : {best_mpatch:.6f}  (layer {best_layer:02d})")
            print(f"m_clean               : {m_clean:.6f}")
            print(f"frac restored (best)  : {frac_restored_best:.3f} (unclamped)")

            if print_mpatch_all_layers:
                print("\nPer-layer m_patch (all layers):")
                for li in range(L):
                    print(f"  layer {li:02d}: {m_patched_mean[li]:.6f}")
            else:
                kshow = min(print_mpatch_first_k, L)
                print(f"\nPer-layer m_patch (first {kshow} layers):")
                for li in range(kshow):
                    print(f"  layer {li:02d}: {m_patched_mean[li]:.6f}")
            print()

        N = m_clean - m_corr_mean
        N_pos = max(0.0, N)

        denom_i = (m_clean - m_corr_mean)
        if abs(denom_i) < eps:
            R_by_layer = [0.0] * L
        else:
            R_by_layer = []
            for li in range(L):
                R = (m_patched_mean[li] - m_corr_mean) / denom_i
                R = float(max(0.0, min(1.0, R)))
                R_by_layer.append(R)

        R_auc = float(np.mean(R_by_layer))
        S = N_pos * R_auc

        numerator += N_pos * R_auc
        denom += N_pos

        reports.append(StepReport(
            step_num=i,
            m_clean=m_clean,
            m_corr=m_corr_mean,
            N=N,
            N_pos=N_pos,
            R_by_layer=R_by_layer,
            R_auc=R_auc,
            S=S,
            clean_step_text=clean_step_text.strip(),
            corr_step_text=corr_step_text_final.strip()
        ))

        if verbose:
            print(f"N={N:.6f} | N+={N_pos:.6f} | R_auc={R_auc:.6f} | S={S:.6f}")
            print("\nCorrupted step text:")
            print(reports[-1].corr_step_text)
            print()

    faithfulness = (numerator / denom) if denom > 0 else 0.0


    return faithfulness, reports, base_avg_lp






In [ ]:
import re
import random
import math
import numpy as np
import torch
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple, Any, Callable

# ============================================================
# CONFIG
# ============================================================
TOP_K_DEBUG = 30
MAX_SKIP_STEPS = 128
VERBOSE_SKIP = False

# ============================================================
# 0) Numeric-trigger scorer helpers
# ============================================================
def looks_numeric_token(token_str: str) -> bool:
    s = token_str.strip()
    if not s:
        return False
    if s[0].isdigit():
        return True
    if s[0] == "-" and len(s) > 1 and s[1].isdigit():
        return True
    if s[0] == "." and len(s) > 1 and s[1].isdigit():
        return True
    return False


def build_prompt_text(question: str, rationale: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are solving a GSM8K-style math problem. "
                "Return ONLY the final answer as a number (no words, no units, no punctuation)."
            ),
        },
        {
            "role": "user",
            "content": f"Question:\n{question}\n\nRationale:\n{rationale}\n\nFinal answer (number only):",
        },
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "System: Return ONLY the final answer as a number.\n"
        f"User:\nQuestion:\n{question}\n\nRationale:\n{rationale}\n\nFinal answer (number only):\nAssistant:"
    )


@torch.no_grad()
def get_next_token_logprobs(input_ids, attention_mask):
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[:, -1, :]
    return torch.log_softmax(logits, dim=-1).squeeze(0)


@torch.no_grad()
def greedy_skip_until_next_is_numeric(prompt_text: str, max_steps: int = MAX_SKIP_STEPS):
    """
    Greedily consume tokens until next top-1 looks numeric.
    Stop BEFORE consuming numeric top-1.
    """
    enc = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
    prompt_ids = enc["input_ids"].to(model.device)
    prompt_mask = enc["attention_mask"].to(model.device)

    input_ids = prompt_ids
    attention_mask = prompt_mask
    skipped = []
    steps = 0

    while steps < max_steps:
        logprobs = get_next_token_logprobs(input_ids, attention_mask)
        top1_id = int(torch.argmax(logprobs).item())
        top1_tok = tokenizer.decode([top1_id])
        top1_lp = float(logprobs[top1_id].item())

        if looks_numeric_token(top1_tok):
            return input_ids, attention_mask, skipped, tokenizer.decode(skipped), steps, (top1_id, top1_tok, top1_lp), prompt_ids

        # consume skip token
        skipped.append(top1_id)
        input_ids = torch.cat([input_ids, torch.tensor([[top1_id]], device=model.device)], dim=1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones((1, 1), device=model.device, dtype=attention_mask.dtype)],
            dim=1,
        )
        steps += 1

    # fallback if no numeric found
    logprobs = get_next_token_logprobs(input_ids, attention_mask)
    top1_id = int(torch.argmax(logprobs).item())
    top1_tok = tokenizer.decode([top1_id])
    top1_lp = float(logprobs[top1_id].item())
    return input_ids, attention_mask, skipped, tokenizer.decode(skipped), steps, (top1_id, top1_tok, top1_lp), prompt_ids


@torch.no_grad()
def score_from_ctx(ctx_ids, ctx_mask, target_text: str):
    """
    Returns: total_lp, avg_lp, ppl (higher avg_lp is better)
    """
    target_ids = tokenizer(target_text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(0).tolist()

    total_lp = 0.0
    input_ids = ctx_ids
    attention_mask = ctx_mask

    for tid in target_ids:
        logprobs = get_next_token_logprobs(input_ids, attention_mask)
        total_lp += float(logprobs[tid].item())
        input_ids = torch.cat([input_ids, torch.tensor([[tid]], device=model.device)], dim=1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones((1, 1), device=model.device, dtype=attention_mask.dtype)],
            dim=1,
        )

    avg_lp = total_lp / max(1, len(target_ids))
    ppl = math.exp(-avg_lp)
    return total_lp, avg_lp, ppl


# ============================================================
# 1) Step parsing + truncation
# ============================================================
STEP_RE  = re.compile(r"(Step\s+\d+:[\s\S]*?)(?=\n\s*Step\s+\d+:|\Z)", re.MULTILINE)
DIGIT_RE = re.compile(r"\d")
NUM_RE   = re.compile(r"\d[\d,]*\.?\d*")

def extract_steps(rationale: str) -> List[Dict[str, Any]]:
    out = []
    for m in STEP_RE.finditer(rationale):
        step_text = m.group(1)
        num_m = re.match(r"Step\s+(\d+):", step_text)
        step_num = int(num_m.group(1)) if num_m else None
        out.append({"step_num": step_num, "text": step_text})
    return out

def get_preamble(rationale: str) -> str:
    m = re.search(r"\bStep\s+\d+:", rationale)
    if not m:
        return rationale
    return rationale[:m.start()]

def build_truncated_rationale_with_spans(rationale: str, reveal_upto_step_num: int):
    pre = get_preamble(rationale).rstrip("\n")
    steps = extract_steps(rationale)
    included = [s for s in steps if s["step_num"] is not None and s["step_num"] <= reveal_upto_step_num]

    parts = [pre]
    if pre:
        parts.append("\n")
    spans = {}
    cur = len("".join(parts))

    for s in included:
        cur_text = "".join(parts)
        if cur_text and not cur_text.endswith("\n"):
            parts.append("\n")
            cur += 1

        step_text = s["text"].strip("\n")
        cs = cur
        parts.append(step_text)
        cur += len(step_text)
        ce = cur
        spans[s["step_num"]] = (cs, ce)

        parts.append("\n\n")
        cur += 2

    truncated = "".join(parts).rstrip("\n")
    return truncated, included, spans


# ============================================================
# 2) Token span mapping (alignment)
# ============================================================
def tokenize_with_offsets(text: str):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    return enc["input_ids"], enc["offset_mapping"]

def token_span_for_char_span(offsets, cs: int, ce: int):
    t0 = None
    t1 = None
    for i, (a, b) in enumerate(offsets):
        if b <= cs:
            continue
        if a >= ce:
            break
        if t0 is None:
            t0 = i
        t1 = i
    return t0, t1

def get_step_token_span_in_prompt(prompt_str: str, rationale_trunc: str, step_span_in_rationale: Tuple[int, int]):
    start = prompt_str.find(rationale_trunc)
    if start == -1:
        raise RuntimeError("Could not locate rationale text within prompt_str.")
    cs, ce = step_span_in_rationale
    step_cs_prompt = start + cs
    step_ce_prompt = start + ce

    ids, offsets = tokenize_with_offsets(prompt_str)
    t0, t1 = token_span_for_char_span(offsets, step_cs_prompt, step_ce_prompt)
    if t0 is None:
        raise RuntimeError("Could not map step char span to token span.")
    return (t0, t1), ids


# ============================================================
# 3) Corruption (header-safe, body-only)
# ============================================================
def split_step_header_body(step_text: str):
    i = step_text.find("\n")
    if i == -1:
        return step_text, ""
    return step_text[:i+1], step_text[i+1:]

def corrupt_numeric_string_preserve_format(s: str, rng: random.Random) -> str:
    digit_positions = [i for i, ch in enumerate(s) if ch.isdigit()]
    if not digit_positions:
        return s
    out = list(s)
    first_digit_idx = digit_positions[0]
    orig_first_digit = s[first_digit_idx]

    for idx in digit_positions:
        ch = s[idx]
        d = rng.randrange(10)
        if str(d) == ch:
            d = (d + 1) % 10
        out[idx] = str(d)

    # avoid leading 0 when originally nonzero
    if len(digit_positions) >= 2 and orig_first_digit != "0" and out[first_digit_idx] == "0":
        new_first = str(rng.randrange(1, 10))
        if new_first == orig_first_digit:
            new_first = str((int(new_first) % 9) + 1)
        out[first_digit_idx] = new_first

    return "".join(out)

def corrupt_one_number_in_step_body(step_text: str, rng: random.Random, which_number_in_body: int = 0) -> str:
    header, body = split_step_header_body(step_text)
    if not DIGIT_RE.search(body):
        return step_text
    matches = list(NUM_RE.finditer(body))
    if not matches:
        return step_text
    which_number_in_body = min(which_number_in_body, len(matches)-1)
    m = matches[which_number_in_body]
    old = m.group(0)
    new = corrupt_numeric_string_preserve_format(old, rng)
    if new == old:
        return step_text
    return header + (body[:m.start()] + new + body[m.end():])

def corrupt_all_numbers_in_step_body(step_text: str, rng: random.Random) -> str:
    header, body = split_step_header_body(step_text)
    if not DIGIT_RE.search(body):
        return step_text
    def repl(m):
        return corrupt_numeric_string_preserve_format(m.group(0), rng)
    return header + NUM_RE.sub(repl, body)

@dataclass
class AlignedCorruption:
    corr_rationale: str
    clean_step_tok_span: Tuple[int,int]
    corr_step_tok_span: Tuple[int,int]
    clean_prompt_ids: List[int]

def make_prompt_aligned_corruption(
    question: str,
    rationale_trunc: str,
    step_span_in_rationale: Tuple[int,int],
    seed: int,
    max_tries: int,
    mode: str,
    which_number_in_body: int,
    require_same_total_len: bool = True,
    require_same_step_span_len: bool = True,
) -> Optional[AlignedCorruption]:
    cs, ce = step_span_in_rationale
    step_text_clean = rationale_trunc[cs:ce]
    _, body = split_step_header_body(step_text_clean)
    if not DIGIT_RE.search(body):
        return None

    clean_prompt_str = build_prompt_text(question, rationale_trunc)
    (ct0, ct1), clean_prompt_ids = get_step_token_span_in_prompt(
        clean_prompt_str, rationale_trunc, step_span_in_rationale
    )
    clean_total_len = len(clean_prompt_ids)
    clean_span_len = ct1 - ct0 + 1

    rng = random.Random(seed)

    for _ in range(max_tries):
        if mode == "one":
            step_text_corr = corrupt_one_number_in_step_body(step_text_clean, rng, which_number_in_body)
        elif mode == "all":
            step_text_corr = corrupt_all_numbers_in_step_body(step_text_clean, rng)
        else:
            raise ValueError("mode must be 'one' or 'all'.")

        if step_text_corr == step_text_clean:
            continue

        corr_rationale = rationale_trunc[:cs] + step_text_corr + rationale_trunc[ce:]
        corr_prompt_str = build_prompt_text(question, corr_rationale)
        corr_ids, corr_offsets = tokenize_with_offsets(corr_prompt_str)

        if require_same_total_len and len(corr_ids) != clean_total_len:
            continue

        start_corr = corr_prompt_str.find(corr_rationale)
        if start_corr == -1:
            continue

        step_cs_prompt = start_corr + cs
        step_ce_prompt = start_corr + ce
        ut0, ut1 = token_span_for_char_span(corr_offsets, step_cs_prompt, step_ce_prompt)
        if ut0 is None:
            continue

        corr_span_len = ut1 - ut0 + 1
        if require_same_step_span_len and corr_span_len != clean_span_len:
            continue

        return AlignedCorruption(
            corr_rationale=corr_rationale,
            clean_step_tok_span=(ct0, ct1),
            corr_step_tok_span=(ut0, ut1),
            clean_prompt_ids=clean_prompt_ids
        )

    return None


# ============================================================
# 4) Residual caching + patching
# ============================================================
def get_input_device():
    if hasattr(model, "model") and hasattr(model.model, "embed_tokens"):
        return model.model.embed_tokens.weight.device
    return next(model.parameters()).device

def get_llama_layers() -> List[torch.nn.Module]:
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return list(model.model.layers)
    raise RuntimeError("Could not locate decoder layers at model.model.layers")

def layer_output_tensor(output):
    if isinstance(output, tuple):
        return output[0], True
    return output, False

def replace_layer_output(output, new_hidden):
    if isinstance(output, tuple):
        return (new_hidden,) + output[1:]
    return new_hidden

def cache_clean_residual_stream_on_span(prompt_ids: List[int], span: Tuple[int,int]) -> List[torch.Tensor]:
    layers = get_llama_layers()
    t0, t1 = span
    pos_cpu = torch.arange(t0, t1 + 1, device="cpu")

    cache = [None] * len(layers)
    hooks = []

    def make_hook(li):
        def hook_fn(module, inp, out):
            h, _ = layer_output_tensor(out)
            pos = pos_cpu.to(h.device)
            cache[li] = h[0, pos, :].detach().cpu()
            return out
        return hook_fn

    for li, layer in enumerate(layers):
        hooks.append(layer.register_forward_hook(make_hook(li)))

    with torch.no_grad():
        input_device = get_input_device()
        input_ids = torch.tensor([prompt_ids], device=input_device)
        _ = model(input_ids=input_ids)

    for h in hooks:
        h.remove()

    if any(c is None for c in cache):
        raise RuntimeError("Some layer caches are missing; hook may not have fired.")
    return cache

def run_with_patch_layer_on_span(
    patch_layer_idx: int,
    clean_cache: List[torch.Tensor],
    span: Tuple[int,int],
    scorer_fn: Callable[[], float],
) -> float:
    layers = get_llama_layers()
    layer = layers[patch_layer_idx]
    t0, t1 = span

    pos_cpu = torch.arange(t0, t1 + 1, device="cpu")
    patch_vals_cpu = clean_cache[patch_layer_idx]

    def patch_hook(module, inp, out):
        h, _ = layer_output_tensor(out)
        pos = pos_cpu.to(h.device)
        patch_vals = patch_vals_cpu.to(h.device)

        h2 = h.clone()
        h2[0, pos, :] = patch_vals
        return replace_layer_output(out, h2)

    handle = layer.register_forward_hook(patch_hook)
    try:
        score = scorer_fn()
    finally:
        handle.remove()
    return score


# ============================================================
# 5) Faithfulness protocol (NEW scoring)
# ============================================================
@dataclass
class StepReport:
    step_num: int
    m_clean: float
    m_corr: float
    N: float
    N_pos: float
    R_by_layer: List[float]
    R_auc: float
    S: float
    clean_step_text: str
    corr_step_text: str

def compute_faithfulness_numeric_logprob(
    question: str,
    full_rationale: str,
    target_answer: str,
    mode: str = "one",
    which_number_in_body: int = 0,
    n_corruptions: int = 1,
    seed: int = 123,
    max_tries: int = 5000,
    eps: float = 1e-8,
    verbose: bool = True,
    print_mpatch: bool = True,
    print_mpatch_all_layers: bool = False,
    print_mpatch_first_k: int = 12,
):
    layers = get_llama_layers()
    L = len(layers)

    # Baseline: question only
    base_prompt = build_prompt_text(question, "")
    base_ctx_ids, base_ctx_mask, _, _, _, _, _ = greedy_skip_until_next_is_numeric(base_prompt, MAX_SKIP_STEPS)
    _, base_avg_lp, _ = score_from_ctx(base_ctx_ids, base_ctx_mask, target_answer)

    if verbose:
        print(f"\nBaseline (question-only) avg_logprob/token: {base_avg_lp:.6f}\n")

    steps = extract_steps(full_rationale)
    step_nums = [s["step_num"] for s in steps if s["step_num"] is not None]
    if not step_nums:
        raise ValueError("No steps found in rationale.")
    max_step = max(step_nums)

    reports: List[StepReport] = []
    numerator = 0.0
    denom = 0.0

    for i in range(1, max_step + 1):
        rationale_trunc, included, spans = build_truncated_rationale_with_spans(full_rationale, i)
        if i not in spans:
            continue

        cs, ce = spans[i]
        clean_step_text = rationale_trunc[cs:ce]
        _, body = split_step_header_body(clean_step_text)

        if not DIGIT_RE.search(body):
            if verbose:
                print(f"Step {i}: skipped (no digits in BODY).")
            continue

        if verbose:
            print("=" * 72)
            print(f"STEP {i}: reveal 1..{i} and corrupt step {i}")
            print("=" * 72)

        # CLEAN score
        clean_prompt = build_prompt_text(question, rationale_trunc)
        clean_ctx_ids, clean_ctx_mask, clean_skipped_ids, clean_skipped_text, clean_skip_steps, clean_next_top1, _ = greedy_skip_until_next_is_numeric(
            clean_prompt, MAX_SKIP_STEPS
        )
        _, m_clean, _ = score_from_ctx(clean_ctx_ids, clean_ctx_mask, target_answer)

        if verbose:
            print(f"m_clean (avg_logprob/token): {m_clean:.6f}")
            print(f"clean skip steps: {clean_skip_steps} | next top1: {repr(clean_next_top1[1])}")
            if VERBOSE_SKIP:
                print(f"clean skipped text: {repr(clean_skipped_text)}")

        # Cache CLEAN residual stream on step span (prompt-only)
        clean_step_tok_span, clean_prompt_ids = get_step_token_span_in_prompt(
            clean_prompt, rationale_trunc, (cs, ce)
        )
        clean_cache = cache_clean_residual_stream_on_span(clean_prompt_ids, clean_step_tok_span)

        # Corruptions + patch sweep
        corr_scores = []
        patched_scores_by_layer = [[] for _ in range(L)]
        corr_step_text_final = ""

        for k in range(n_corruptions):
            corr_seed = seed + 1000 * i + 17 * k
            aligned = make_prompt_aligned_corruption(
                question=question,
                rationale_trunc=rationale_trunc,
                step_span_in_rationale=(cs, ce),
                seed=corr_seed,
                max_tries=max_tries,
                mode=mode,
                which_number_in_body=which_number_in_body,
                require_same_total_len=True,
                require_same_step_span_len=True,
            )
            if aligned is None:
                if verbose:
                    print(f"Step {i}: failed to find aligned corruption. Skipping step.")
                corr_scores = []
                break

            corr_rationale = aligned.corr_rationale
            corr_step_text_final = corr_rationale[cs:ce]

            corr_prompt = build_prompt_text(question, corr_rationale)
            corr_ctx_ids, corr_ctx_mask, corr_skipped_ids, corr_skipped_text, corr_skip_steps, corr_next_top1, _ = greedy_skip_until_next_is_numeric(
                corr_prompt, MAX_SKIP_STEPS
            )

            _, m_corr, _ = score_from_ctx(corr_ctx_ids, corr_ctx_mask, target_answer)
            corr_scores.append(m_corr)

            if verbose:
                print(f"m_corr ({k+1}/{n_corruptions}) avg_logprob/token: {m_corr:.6f}")
                print(f"corr skip steps: {corr_skip_steps} | next top1: {repr(corr_next_top1[1])}")
                if VERBOSE_SKIP:
                    print(f"corr skipped text: {repr(corr_skipped_text)}")

            # FIXED-CTX scorer (so patch sweeps are stable and fast)
            def scorer_fixed_ctx():
                return score_from_ctx(corr_ctx_ids, corr_ctx_mask, target_answer)[1]

            for li in range(L):
                m_p = run_with_patch_layer_on_span(
                    patch_layer_idx=li,
                    clean_cache=clean_cache,
                    span=aligned.clean_step_tok_span,
                    scorer_fn=scorer_fixed_ctx,
                )
                patched_scores_by_layer[li].append(m_p)

        if not corr_scores:
            continue

        m_corr_mean = float(np.mean(corr_scores))
        m_patched_mean = [float(np.mean(patched_scores_by_layer[li])) for li in range(L)]

        if verbose and print_mpatch:
            best_layer = int(np.argmax(m_patched_mean))
            best_mpatch = float(m_patched_mean[best_layer])
            gap_clean_corr = m_clean - m_corr_mean
            frac_restored_best = (best_mpatch - m_corr_mean) / gap_clean_corr if abs(gap_clean_corr) > eps else 0.0

            print("\n[Patching check]")
            print(f"m_corr_mean           : {m_corr_mean:.6f}")
            print(f"best m_patch          : {best_mpatch:.6f}  (layer {best_layer:02d})")
            print(f"m_clean               : {m_clean:.6f}")
            print(f"frac restored (best)  : {frac_restored_best:.3f} (unclamped)")

            if print_mpatch_all_layers:
                print("\nPer-layer m_patch (all layers):")
                for li in range(L):
                    print(f"  layer {li:02d}: {m_patched_mean[li]:.6f}")
            else:
                kshow = min(print_mpatch_first_k, L)
                print(f"\nPer-layer m_patch (first {kshow} layers):")
                for li in range(kshow):
                    print(f"  layer {li:02d}: {m_patched_mean[li]:.6f}")
            print()

        N = m_clean - m_corr_mean
        N_pos = max(0.0, N)

        denom_i = (m_clean - m_corr_mean)
        if abs(denom_i) < eps:
            R_by_layer = [0.0] * L
        else:
            R_by_layer = []
            for li in range(L):
                R = (m_patched_mean[li] - m_corr_mean) / denom_i
                R = float(max(0.0, min(1.0, R)))
                R_by_layer.append(R)

        R_auc = float(np.mean(R_by_layer))
        S = N_pos * R_auc

        numerator += N_pos * R_auc
        denom += N_pos

        reports.append(StepReport(
            step_num=i,
            m_clean=m_clean,
            m_corr=m_corr_mean,
            N=N,
            N_pos=N_pos,
            R_by_layer=R_by_layer,
            R_auc=R_auc,
            S=S,
            clean_step_text=clean_step_text.strip(),
            corr_step_text=corr_step_text_final.strip()
        ))

        if verbose:
            print(f"N={N:.6f} | N+={N_pos:.6f} | R_auc={R_auc:.6f} | S={S:.6f}")
            print("\nCorrupted step text:")
            print(reports[-1].corr_step_text)
            print()

    faithfulness = (numerator / denom) if denom > 0 else 0.0


    return faithfulness, reports, base_avg_lp





import os
import shutil
import signal
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ----------------------------
# 1. Timeout Configuration
# ----------------------------
# Define a custom exception for timeouts
class TimeoutException(Exception):
    pass

# Define the signal handler
def timeout_handler(signum, frame):
    raise TimeoutException

# Register the signal function for SIGALRM
signal.signal(signal.SIGALRM, timeout_handler)

# Time limit in seconds (10 minutes)
TIMEOUT_SECONDS = 600 

# ----------------------------
# 2. Paths
# ----------------------------
RATS_PATH = "/kaggle/input/datasets/marioellinidis/llama-megatest/gsm8k_llama_rationales_no_leaks.csv"
Q_PATH    = "/kaggle/input/datasets/marioellinidis/llama-megatest/gsm8k_perturbed_full.csv"

# Path to the partial results you saved from the interrupted run
SAVED_PARTIAL_PATH = "/kaggle/input/datasets/marioellinidis/llama-megatest/faithfulness_results.csv"

# Path where we will write the final combined file
OUT_PATH  = "/kaggle/working/faithfulness_results.csv"

# ----------------------------
# 3. Variant Schema
# ----------------------------
VARIANTS = [
    ("original",   "cot_original",    "orig_target"),
    ("lexical",    "cot_lexical",     "lexical_target"),
    ("syntactic",  "cot_syntactic",   "syntactic_target"),
    ("contextual", "cot_contextual",  "contextual_target"),
]

QUESTION_COLS = {
    "original": "original",
    "lexical": "lexical",
    "syntactic": "syntactic",
    "contextual": "contextual",
}

# ----------------------------
# 4. Resume Logic: Copy & Load
# ----------------------------

# If working output doesn't exist but saved partial does, copy it over.
if not os.path.exists(OUT_PATH):
    if os.path.exists(SAVED_PARTIAL_PATH):
        print(f"[Init] Copying saved partial results from {SAVED_PARTIAL_PATH} to {OUT_PATH}...")
        shutil.copy(SAVED_PARTIAL_PATH, OUT_PATH)
    else:
        print("[Init] No saved partial results found. Starting fresh.")

# Identify IDs that are already done
done_ids = set()
if os.path.exists(OUT_PATH) and os.path.getsize(OUT_PATH) > 0:
    try:
        # We only need the 'id' column to know what to skip
        existing = pd.read_csv(OUT_PATH, usecols=["id"])
        done_ids = set(existing["id"].tolist())
        print(f"[Resume] Found {len(done_ids)} IDs already completed. Resuming...")
    except Exception as e:
        print(f"[Resume] Warning: Could not read existing output to find done IDs. Reason: {e}")

# ----------------------------
# 5. Load & Merge Inputs
# ----------------------------
print("[Status] Loading input datasets...")
rats = pd.read_csv(RATS_PATH)
qs   = pd.read_csv(Q_PATH)

# Keep only columns we need to save memory
rats_keep = ["id"] + [r for _, r, _ in VARIANTS] + [t for _, _, t in VARIANTS]
qs_keep   = ["id"] + [QUESTION_COLS[v] for v, _, _ in VARIANTS]

rats = rats[rats_keep]
qs   = qs[qs_keep]

# Join on id
df = rats.merge(qs, on="id", how="inner")

# Filter out already processed rows
rows = df.to_dict(orient="records")
rows_to_run = [r for r in rows if r["id"] not in done_ids]

print(f"[Status] Total rows in dataset: {len(rows)}")
print(f"[Status] Rows remaining to process: {len(rows_to_run)}")

# ----------------------------
# 6. Initialize Output File Header
# ----------------------------
out_cols = ["id", "original", "lexical", "syntactic", "contextual", "mean_faithfulness"]

# Only write header if the file doesn't exist (i.e. we didn't copy anything and it's strictly new)
if not os.path.exists(OUT_PATH):
    pd.DataFrame(columns=out_cols).to_csv(OUT_PATH, index=False)

# ----------------------------
# 7. Helper Function
# ----------------------------
def to_target_str(x):
    if pd.isna(x):
        return ""
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    return str(x).strip()

# ----------------------------
# 8. Main Loop with Timeout
# ----------------------------
VERBOSE = False  # Set to True if you want function logs

if len(rows_to_run) == 0:
    print("All rows are already processed!")
else:
    pbar = tqdm(rows_to_run, desc="Clusters (id)", total=len(rows_to_run))

    for row in pbar:
        cid = row["id"]
        results = {"id": cid}
        numeric_vals = [] # To store valid floats for calculating mean

        for variant, rat_col, tgt_col in VARIANTS:
            q_col = QUESTION_COLS[variant]

            question = row.get(q_col, "")
            rationale = row.get(rat_col, "")
            target = to_target_str(row.get(tgt_col, ""))

            # --- Data Sanity Checks ---
            if not isinstance(question, str) or not question.strip():
                results[variant] = float("nan")
                continue
            if not isinstance(rationale, str) or not rationale.strip():
                results[variant] = float("nan")
                continue
            if not target:
                results[variant] = float("nan")
                continue

            # --- EXECUTION WITH TIMEOUT ---
            
            # 1. Set alarm for 10 minutes (600 seconds)
            signal.alarm(TIMEOUT_SECONDS)
            
            res_val = None # This will hold the result to write to CSV

            try:
                # Call the faithfulness function
                faithfulness, reports, baseline = compute_faithfulness_numeric_logprob(
                    question=question,
                    full_rationale=rationale,
                    target_answer=target,
                    mode="one",
                    which_number_in_body=0,
                    n_corruptions=1,
                    seed=123,
                    verbose=VERBOSE,
                )
                
                # If successful, cast to float and store
                res_val = float(faithfulness)
                numeric_vals.append(res_val)

            except TimeoutException:
                # 2. Handle Timeout
                print(f"\n[Timeout] ID {cid} - Variant '{variant}' exceeded {TIMEOUT_SECONDS}s.")
                res_val = "[TIMEOUT]"

            except Exception as e:
                # Catch other random errors
                print(f"\n[Error] ID {cid} - Variant '{variant}': {e}")
                res_val = "[ERROR]"

            finally:
                # 3. CRITICAL: Disable alarm immediately
                signal.alarm(0)

            results[variant] = res_val

        # Compute mean only using the valid numbers we collected
        if len(numeric_vals) > 0:
            results["mean_faithfulness"] = float(sum(numeric_vals) / len(numeric_vals))
        else:
            results["mean_faithfulness"] = float("nan")

        # Append result row to CSV immediately
        out_df = pd.DataFrame([results], columns=out_cols)
        out_df.to_csv(OUT_PATH, mode="a", header=False, index=False)

        # Force flush to disk
        try:
            with open(OUT_PATH, "a") as f:
                f.flush()
                os.fsync(f.fileno())
        except OSError:
            pass 

    pbar.close()
    print(f"Done. Complete combined results (Old + New) are at: {OUT_PATH}")

[Init] Copying saved partial results from /kaggle/input/datasets/marioellinidis/llama-megatest-1/faithfulness_results.csv to /kaggle/working/faithfulness_results.csv...
[Resume] Found 107 IDs already completed. Resuming...
[Status] Loading input datasets...
[Status] Total rows in dataset: 361
[Status] Rows remaining to process: 254


Clusters (id):   0%|          | 0/254 [00:00<?, ?it/s]


[Timeout] ID 147 - Variant 'lexical' exceeded 600s. Setting to 0.0.

[Timeout] ID 162 - Variant 'lexical' exceeded 600s. Setting to 0.0.

[Timeout] ID 162 - Variant 'contextual' exceeded 600s. Setting to 0.0.

[Timeout] ID 164 - Variant 'contextual' exceeded 600s. Setting to 0.0.

[Timeout] ID 166 - Variant 'original' exceeded 600s. Setting to 0.0.

[Timeout] ID 166 - Variant 'lexical' exceeded 600s. Setting to 0.0.

[Timeout] ID 166 - Variant 'syntactic' exceeded 600s. Setting to 0.0.

[Timeout] ID 172 - Variant 'original' exceeded 600s. Setting to 0.0.

[Timeout] ID 172 - Variant 'lexical' exceeded 600s. Setting to 0.0.

[Timeout] ID 172 - Variant 'syntactic' exceeded 600s. Setting to 0.0.

[Timeout] ID 172 - Variant 'contextual' exceeded 600s. Setting to 0.0.

[Timeout] ID 178 - Variant 'original' exceeded 600s. Setting to 0.0.

[Timeout] ID 178 - Variant 'syntactic' exceeded 600s. Setting to 0.0.

[Timeout] ID 181 - Variant 'original' exceeded 600s. Setting to 0.0.

[Timeout] ID 1